# GeometricNearestNeighbors usage examples

Anton Antonov   
June 2026

---

## Introduction

This notebook has the most typical workflow of using the Python package "GeometricNearestNeighbors", [AAp1].

---

## Setup

In [1]:
from GeometricNearestNeighborsProcessor import *
from RandomDataGenerators import *
from OutlierIdentifiers import *

import numpy
import random

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

---

## Usage examples

Generate random points:

In [2]:
dfPoints = random_data_frame(n_rows=30, columns_spec = ["X", "Y"], generators= {"X": numpy.random.normal, "Y": numpy.random.normal})
print(dfPoints.shape)
print(dfPoints[1:6])

(30, 2)
          X         Y
1 -0.651435  0.119060
2  0.563057  0.107302
3  0.688770 -0.006726
4 -0.213794  0.129886
5 -1.137647  0.955632


Here is a summary:

In [3]:
dfPoints.describe()

,X,Y
count,30.000000,30.000000
mean,-0.122722,0.026645
std,1.107593,1.048453
min,-2.007130,-2.421242
25%,-0.901688,-0.325681
50%,-0.158129,0.108314
75%,0.518388,0.444806
max,2.732892,2.628951


Here is a plot of the points:

In [4]:
fig = px.scatter(dfPoints, x="X", y="Y", template="plotly_dark")
fig.show()

A typical pipeline of geometric nearest neighbors processing:

In [5]:
gnnObj = (GeometricNearestNeighborsProcessor(dfPoints)
   .make_nearest_function(distance_function = "EuclideanDistance")
   .compute_thresholds(number_of_nearest_neighbors = 10, aggregation_function = "mean", outlier_identifier = "QuartileIdentifierParameters")
   .find_anomalies()
   .echo_function_value("Anomaly points:")
)

Anomaly points:
          X         Y    Radius
0 -1.853340 -2.421242  1.922563
1  2.732892 -1.055796  2.013650
2  1.345801  2.628951  2.064770


One way to plot the anomalies together with the data points using the package "plotly" is to merge into one data frame and add an indicator column:

In [6]:
# Assign data
df1 = gnnObj.take_data()
df2 = gnnObj.take_value()

# Mark points that are in df2
df1['highlight'] = df1.merge(df2.assign(flag=1), on=['X', 'Y'], how='left')['flag'].fillna(0)

# Convert to category for plotting
df1['highlight'] = df1['highlight'].map({0: 'data', 1: 'anomalies'})

# Plot
fig = px.scatter(df1, x='X', y='Y', color='highlight',
                 color_discrete_map={'data': 'blue', 'anomalies': 'red'},
                 template="plotly_dark")

fig.show()

Alternatively, the anomaly points can be shown as smaller, in red, and on top of the data points:

In [7]:
fig = go.Figure()

# All points (larger)
fig.add_trace(go.Scatter(
    x=df1['X'],
    y=df1['Y'],
    mode='markers',
    marker=dict(size=12, color='blue'),
    name='data'
))

# Anomaly points (smaller, on top)
fig.add_trace(go.Scatter(
    x=df2['X'],
    y=df2['Y'],
    mode='markers',
    marker=dict(size=6, color='red', line=dict(width=1, color='black')),
    name='anomalies'
))

fig.update_layout(
    template="plotly_dark",
    xaxis_title="X Axis",
    yaxis_title="Y Axis"
)

fig.show()

The following data can be used :

```python
aPoints = [(r["X"], r["Y"]) for r in dfPoints.to_dict(orient="records")]
aPoints = dict(zip(random_word(len(aPoints)), aPoints))
```

With that the plots above have to use the columns "x1" and "x2". Or the attribute "data" of `gnnObj` have to be changed to have the columns "X" and "Y".

----

## Nearest neighbors

Here we generate another set of random points using the same random point generators:

In [8]:
dfPoints2 = random_data_frame(n_rows=40, columns_spec = ["X", "Y"], generators= {"X": numpy.random.normal, "Y": numpy.random.normal})
print(dfPoints2.shape)

(40, 2)


For each point of the second set find its top-3 nearest neighbors in the first set:

In [9]:
dfNNs = pd.concat(
    [
        gnnObj.find_nearest(point=[row["X"], row["Y"]], n=3)
        .take_value()
        .assign(SearchIndex=i)
        for i, row in enumerate(dfPoints2.to_dict(orient="records"))
    ],
    ignore_index=True
)

dfNNs

,Index,ID,Distance,SearchIndex
0,1,p01,0.451318,0
1,17,p17,0.532140,0
2,5,p05,0.570540,0
3,29,p29,0.240624,1
4,25,p25,0.380394,1
...,...,...,...,...
115,17,p17,0.528243,38
116,5,p05,0.591152,38
117,16,p16,0.164303,39
118,22,p22,0.588009,39


----

## Classification

Here the points of second set are classified into being anomalous or not:

In [10]:
gnnObj.classify(dfPoints2).take_value()

,Index,Radius,Label
0,0,0.701211,True
1,1,0.690686,True
2,2,0.850617,True
3,3,1.331449,True
4,4,0.476110,True
5,5,0.619807,True
6,6,0.687441,True
7,7,0.938066,True
8,8,1.164826,True
9,9,0.395376,True


Plot the original data points (in gray) and the new data points by marking the anomalous ones:

In [11]:
df0 = gnnObj.take_data()

dfClassified = gnnObj.classify(dfPoints2).take_value()
dfCombined = dfPoints2.join(dfClassified)

df1 = dfCombined[dfCombined["Label"] == True]
df2 = dfCombined[dfCombined["Label"] == False]

fig = go.Figure()

# Original data points
fig.add_trace(go.Scatter(
    x=df0['X'],
    y=df0['Y'],
    mode='markers',
    marker=dict(size=8, color='gray'),
    name='data'
))

# Non-anomaly points
fig.add_trace(go.Scatter(
    x=df1['X'],
    y=df1['Y'],
    mode='markers',
    marker=dict(size=10, color='blue', line=dict(width=1, color='black')),
    name='non-anomalies'
))

# Anomaly points
fig.add_trace(go.Scatter(
    x=df2['X'],
    y=df2['Y'],
    mode='markers',
    marker=dict(size=10, color='red', line=dict(width=1, color='black')),
    name='anomalies'
))

fig.update_layout(
    template="plotly_dark",
    xaxis_title="X Axis",
    yaxis_title="Y Axis"
)

fig.show()

----

## Proximity matrix

Here is the proximity matrix:

In [14]:
mat=gnnObj.compute_proximity_matrix().take_value()
mat

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 330 stored elements and shape (30, 30)>

Here is a corresponding matrix plot:

In [16]:
rows, cols = mat.nonzero()

fig = go.Figure(data=go.Scattergl(
    x=cols,
    y=rows,
    mode='markers',
    marker=dict(size=3)
))

fig.update_layout(
    title="Proximity matrix",
    xaxis_title="Columns",
    yaxis_title="Rows",
    yaxis_autorange='reversed',
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(constrain='domain'),
    width=600,
    height=600,
    template="plotly_dark"
)

fig.show()

---

## References

[AAp1] Anton Antonov, [GeometricNearestNeighborsProcessor](https://github.com/antononcube/Python-GeometricNearestNeighborsProcessor), Python package, (2026), [GitHub/antononcube](https://github.com/antononcube).([PIPy.org](https://pypi.org/project/GeometricNearestNeighborsProcessor/).)

[AAp2] Anton Antonov, [RandomDataGenerators](https://github.com/antononcube/Python-packages/tree/main/RandomDataGenerators), Python package, (2021-2026), [GitHub/antononcube](https://github.com/antononcube). ([PIPy.org](https://pypi.org/project/RandomDataGenerators/).)

[AAp3] Anton Antonov, [OutlierIdentifiers](https://github.com/antononcube/Python-packages/tree/main/OutlierIdentifiers), Python package, (2024), [GitHub/antononcube](https://github.com/antononcube). ([PIPy.org](https://pypi.org/project/OutlierIdentifiers/).)